Google Stock (GOOGL) Time Series Forecasting
=============================================
Models covered : AR, MA, ARIMA, SARIMA, SARIMAX
Libraries used  : numpy, pandas, matplotlib, statsmodels

Pipeline:
1. Load & prepare data
2. Train / test split
3. Fit AR, MA, ARIMA, SARIMA, SARIMAX models
4. Forecast on the test horizon
5. Plot actual vs forecast for every model
6. Evaluate models (RMSE / MAE)
7. Export all forecasts to a CSV file


In [1]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless backend (saving figures, not showing)
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error



---


In [2]:
# 1. LOAD & PREPARE DATA


---


In [5]:
DATA_PATH = "C:/Users/Administrator/Google Stock data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df.set_index("Date", inplace=True)



Business-day frequency + fill any gaps (weekends/holidays already absent,
but reindexing keeps statsmodels' date-based indexing happy)


In [6]:
df = df.asfreq("B")
df["Close_GOOGL"] = df["Close_GOOGL"].ffill()
df["Volume_GOOGL"] = df["Volume_GOOGL"].ffill()

series = df["Close_GOOGL"]

# Scale volume (millions) so its magnitude doesn't destabilize the MLE optimizer
exog_full = df[["Volume_GOOGL"]].copy()
exog_full["Volume_GOOGL"] = exog_full["Volume_GOOGL"] / 1_000_000.0

print(f"Data range : {df.index.min().date()}  ->  {df.index.max().date()}")
print(f"Total observations: {len(df)}")



Data range : 2021-08-18  ->  2026-08-14
Total observations: 1303


---


In [7]:
# 2. TRAIN / TEST SPLIT  (last 30 business days held out for testing)


---


In [8]:
FORECAST_HORIZON = 30

train = series.iloc[:-FORECAST_HORIZON]
test  = series.iloc[-FORECAST_HORIZON:]

exog_train = exog_full.iloc[:-FORECAST_HORIZON]
exog_test  = exog_full.iloc[-FORECAST_HORIZON:]

print(f"Train size: {len(train)}  |  Test size: {len(test)}")

results = {}          # model_name -> forecast Series
metrics  = {}          # model_name -> dict(rmse, mae)


def evaluate(name, forecast):
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    metrics[name] = {"RMSE": rmse, "MAE": mae}
    results[name] = forecast
    print(f"[{name:8s}] RMSE = {rmse:8.4f}   MAE = {mae:8.4f}")




Train size: 1273  |  Test size: 30


---


In [9]:
# 3a. AR MODEL  (AutoRegressive)


---


In [10]:
print("\n--- Fitting AR model ---")
ar_lags = 15
ar_model = AutoReg(train, lags=ar_lags, old_names=False).fit()
ar_forecast = ar_model.predict(start=len(train), end=len(train) + FORECAST_HORIZON - 1)
ar_forecast.index = test.index
evaluate("AR", ar_forecast)




--- Fitting AR model ---
[AR      ] RMSE =  22.4072   MAE =  18.0222


---


In [11]:
# 3b. MA MODEL  (Moving Average -> ARIMA(0,0,q))


---


In [12]:
print("\n--- Fitting MA model ---")
ma_order = (0, 0, 5)
ma_model = ARIMA(train, order=ma_order).fit()
ma_forecast = ma_model.forecast(steps=FORECAST_HORIZON)
ma_forecast.index = test.index
evaluate("MA", ma_forecast)




--- Fitting MA model ---
[MA      ] RMSE = 169.2882   MAE = 163.9307


---


In [13]:
# 3c. ARIMA MODEL  (AutoRegressive Integrated Moving Average)


---


In [14]:
print("\n--- Fitting ARIMA model ---")
arima_order = (5, 1, 2)
arima_model = ARIMA(train, order=arima_order).fit()
arima_forecast = arima_model.forecast(steps=FORECAST_HORIZON)
arima_forecast.index = test.index
evaluate("ARIMA", arima_forecast)




--- Fitting ARIMA model ---


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[ARIMA   ] RMSE =  17.3650   MAE =  13.2065


---


In [15]:
# 3d. SARIMA MODEL  (Seasonal ARIMA -> SARIMAX without exogenous vars)


---


In [16]:
print("\n--- Fitting SARIMA model ---")
sarima_order = (2, 1, 2)
sarima_seasonal_order = (1, 1, 1, 5)   # weekly (5 business-day) seasonality
sarima_model = SARIMAX(
    train,
    order=sarima_order,
    seasonal_order=sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
sarima_forecast = sarima_model.forecast(steps=FORECAST_HORIZON)
sarima_forecast.index = test.index
evaluate("SARIMA", sarima_forecast)




--- Fitting SARIMA model ---


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[SARIMA  ] RMSE =  21.2911   MAE =  17.0780


---


In [17]:
# 3e. SARIMAX MODEL  (SARIMA + exogenous regressor: trading Volume)


---


In [18]:
print("\n--- Fitting SARIMAX model ---")
sarimax_order = (2, 1, 2)
sarimax_seasonal_order = (1, 1, 1, 5)
sarimax_model = SARIMAX(
    train,
    exog=exog_train,
    order=sarimax_order,
    seasonal_order=sarimax_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False, method="powell", maxiter=200)
sarimax_forecast = sarimax_model.forecast(steps=FORECAST_HORIZON, exog=exog_test)



--- Fitting SARIMAX model ---


Safety clip: keep forecast within a sane band relative to recent price levels
in case the exogenous-driven model still overshoots on out-of-sample volume.


In [19]:
recent_band = train.iloc[-60:]
lo, hi = recent_band.min() * 0.5, recent_band.max() * 1.5
sarimax_forecast = sarimax_forecast.clip(lower=lo, upper=hi)
sarimax_forecast.index = test.index
evaluate("SARIMAX", sarimax_forecast)



[SARIMAX ] RMSE =  21.2063   MAE =  17.0147


---


In [20]:
# 4. PLOTS  (actual vs forecast for every model)


---


In [21]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

# Full history + zoom-in context: show last 120 train points + test window
context = 120
plot_train_tail = train.iloc[-context:]

for ax, (name, fc) in zip(axes, results.items()):
    ax.plot(plot_train_tail.index, plot_train_tail.values, label="Train (recent)", color="steelblue")
    ax.plot(test.index, test.values, label="Actual (test)", color="black", linewidth=2)
    ax.plot(fc.index, fc.values, label=f"{name} forecast", color="crimson", linestyle="--", marker="o", markersize=3)
    ax.set_title(f"{name} Model  |  RMSE={metrics[name]['RMSE']:.2f}  MAE={metrics[name]['MAE']:.2f}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close Price ($)")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)

# hide the unused 6th subplot
for extra_ax in axes[len(results):]:
    extra_ax.axis("off")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
print("\nSaved plot: model_comparison.png")

# Combined single-plot overview
plt.figure(figsize=(14, 7))
plt.plot(plot_train_tail.index, plot_train_tail.values, label="Train (recent)", color="steelblue")
plt.plot(test.index, test.values, label="Actual", color="black", linewidth=2.5)
colors = ["crimson", "orange", "green", "purple", "brown"]
for (name, fc), c in zip(results.items(), colors):
    plt.plot(fc.index, fc.values, label=f"{name} forecast", linestyle="--", color=c)
plt.title("GOOGL Close Price — All Models Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("Close Price ($)")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("all_models_overview.png", dpi=150)
print("Saved plot: all_models_overview.png")




Saved plot: model_comparison.png
Saved plot: all_models_overview.png


---


In [22]:
# 5. METRICS SUMMARY


---


In [ ]:
metrics_df = pd.DataFrame(metrics).T
metrics_df.index.name = "Model"
print("\n=== Model Performance Summary ===")
print(metrics_df.sort_values("RMSE"))
metrics_df.to_csv("model_metrics.csv")



---


In [ ]:
# 6. EXPORT FORECASTED DATA TO CSV


---


In [ ]:
forecast_df = pd.DataFrame({"Date": test.index, "Actual_Close": test.values})
for name, fc in results.items():
    forecast_df[f"{name}_Forecast"] = fc.values

forecast_df.to_csv("forecasted_data.csv", index=False)
print("\nSaved forecasted data: forecasted_data.csv")
print(forecast_df.head())



---


In [ ]:
# 7. FUTURE FORECAST (beyond available data) using the best model (lowest RMSE)


---


In [ ]:
best_model_name = metrics_df["RMSE"].idxmin()
print(f"\nBest model based on RMSE: {best_model_name}")

FUTURE_STEPS = 15
future_index = pd.bdate_range(start=series.index[-1] + pd.Timedelta(days=1), periods=FUTURE_STEPS)

if best_model_name == "AR":
    full_ar = AutoReg(series, lags=ar_lags, old_names=False).fit()
    future_forecast = full_ar.predict(start=len(series), end=len(series) + FUTURE_STEPS - 1)
elif best_model_name == "MA":
    full_ma = ARIMA(series, order=ma_order).fit()
    future_forecast = full_ma.forecast(steps=FUTURE_STEPS)
elif best_model_name == "ARIMA":
    full_arima = ARIMA(series, order=arima_order).fit()
    future_forecast = full_arima.forecast(steps=FUTURE_STEPS)
elif best_model_name == "SARIMA":
    full_sarima = SARIMAX(series, order=sarima_order, seasonal_order=sarima_seasonal_order,
                           enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    future_forecast = full_sarima.forecast(steps=FUTURE_STEPS)
else:  # SARIMAX
    last_vol = exog_full["Volume_GOOGL"].iloc[-30:].mean()   # already scaled (millions)
    future_exog = pd.DataFrame({"Volume_GOOGL": [last_vol] * FUTURE_STEPS}, index=future_index)
    full_sarimax = SARIMAX(series, exog=exog_full, order=sarimax_order, seasonal_order=sarimax_seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, method="powell", maxiter=200)
    future_forecast = full_sarimax.forecast(steps=FUTURE_STEPS, exog=future_exog)
    band_lo, band_hi = series.iloc[-60:].min() * 0.5, series.iloc[-60:].max() * 1.5
    future_forecast = future_forecast.clip(lower=band_lo, upper=band_hi)

future_forecast.index = future_index
future_df = pd.DataFrame({"Date": future_index, f"{best_model_name}_Future_Forecast": future_forecast.values})
future_df.to_csv("future_forecast.csv", index=False)
print(f"\nSaved future ({FUTURE_STEPS}-day) forecast: future_forecast.csv")
print(future_df)

print("\nDone.")
